# Airbnb Paris – Exp 3: Semantische Relevanz (SHAP)
- Modell: SAP ConTextTab (verarbeitet Zahlen **und** Freitexte nativ, binäre Klassifikation)
- SHAP (KernelExplainer) je Feature; Aggregation numerisch vs. Freitext

In [1]:
import numpy as np
import pandas as pd
import mlflow
import shap
import torch  # noqa: F401  (vor sap_rpt_oss laden: TORCH_LIBRARY-Doppelregistrierung vermeiden)
from sap_rpt_oss import SAP_RPT_OSS_Classifier

## Daten, Subsample & ConTextTab
- `cleaned_text` (numerisch + Freitext); balancierter 1:4-Kontext (120/480) zum Fitten, 30 Zeilen zum Erklären

In [2]:
LABEL = "is_top_rating"
TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]

df = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
X = df.drop(columns=[LABEL])
y = df[LABEL].astype(int)
num_cols = [c for c in X.columns if c not in TEXT_COLS]
outlier_label = y.value_counts().idxmin()  # = 0 (rating <= 3)

rng = np.random.RandomState(42)
out_idx = y.index[y == outlier_label].to_numpy()
in_idx = y.index[y != outlier_label].to_numpy()
rng.shuffle(out_idx)
rng.shuffle(in_idx)
ctx_id = np.concatenate([out_idx[:120], in_idx[:480]])
explain_id = np.concatenate([out_idx[120:130], in_idx[480:500]])  # 10 Outlier + 20 Inlier

ctx = SAP_RPT_OSS_Classifier(max_context_size=len(ctx_id), bagging=1)
ctx.fit(X.loc[ctx_id], y.loc[ctx_id].values)
outlier_col = sorted(np.unique(y.loc[ctx_id]).tolist()).index(outlier_label)
print("Kontext:", len(ctx_id), "| erklärt:", len(explain_id), "| Features:", X.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Kontext: 600 | erklärt: 30 | Features: 44


## SHAP (KernelExplainer, modell-agnostisch)
- Erklärt P(Outlier); je Feature ein SHAP-Wert (jede Freitextspalte = 1 Feature)

In [3]:
def predict_outlier(arr):
    d = pd.DataFrame(arr, columns=X.columns)
    d[num_cols] = d[num_cols].astype(float)
    return ctx.predict_proba(d)[:, outlier_col]

background = X.loc[ctx_id].sample(10, random_state=42)
explain = X.loc[explain_id]
sv = shap.KernelExplainer(predict_outlier, background).shap_values(explain, nsamples=100)
imp = pd.Series(np.abs(np.array(sv)).reshape(-1, X.shape[1]).mean(axis=0), index=X.columns)

  0%|          | 0/30 [00:00<?, ?it/s]

## Ergebnis: SHAP je Feature + numerisch vs. Freitext

In [4]:
table = imp.sort_values(ascending=False).round(5).to_frame("mean_abs_shap")
numeric_total = float(imp[num_cols].sum())
text_total = float(imp[TEXT_COLS].sum())
display(table)
print(f"numerisch gesamt={numeric_total:.5f}  |  freitext gesamt={text_total:.5f}")

,mean_abs_shap
neighborhood_overview,0.00809
host_about,0.00711
name,0.00554
description,0.00516
longitude,0.00382
amenities_count,0.00342
host_tenure_days,0.00337
host_response_time_within_a_day,0.00333
host_response_time_unknown,0.00320
host_response_rate,0.00242


numerisch gesamt=0.05079  |  freitext gesamt=0.02590


## Loggen

In [5]:
mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_3")
with mlflow.start_run(run_name="shap_contexttab"):
    for c in TEXT_COLS:
        mlflow.log_metric(f"text_{c}", float(imp[c]))
    mlflow.log_metric("numeric_total", numeric_total)
    mlflow.log_metric("text_total", text_total)

/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
